# Ride-Sharing Demand & Revenue Intelligence  

## Business Objective

The goal of this project is to analyze NYC taxi ride data to understand demand patterns, revenue distribution, and prepare the dataset for dynamic surge pricing simulation.

This notebook focuses on:
- Data cleaning
- Outlier removal
- Feature engineering
- Preparing a clean dataset for analysis

## Data Cleaning & Feature Engineering

In [1]:
import pandas as pd 
import numpy as np 
import matplotlib.pyplot as plt 
import seaborn as sns 

pd.set_option('display.max_columns', None)

## Dataset Loading........

In [2]:
df = pd.read_parquet(r'C:\prerit\Projects\Ride-Sharing-Demand-Intelligence\Raw\yellow_tripdata_2025-01.parquet')
df.head()

,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee,cbd_congestion_fee
0,1,2025-01-01 00:18:38,2025-01-01 00:26:59,1.0,1.60,1.0,N,229,237,1,10.0,3.5,0.5,3.00,0.0,1.0,18.00,2.5,0.0,0.0
1,1,2025-01-01 00:32:40,2025-01-01 00:35:13,1.0,0.50,1.0,N,236,237,1,5.1,3.5,0.5,2.02,0.0,1.0,12.12,2.5,0.0,0.0
2,1,2025-01-01 00:44:04,2025-01-01 00:46:01,1.0,0.60,1.0,N,141,141,1,5.1,3.5,0.5,2.00,0.0,1.0,12.10,2.5,0.0,0.0
3,2,2025-01-01 00:14:27,2025-01-01 00:20:01,3.0,0.52,1.0,N,244,244,2,7.2,1.0,0.5,0.00,0.0,1.0,9.70,0.0,0.0,0.0
4,2,2025-01-01 00:21:34,2025-01-01 00:25:06,3.0,0.66,1.0,N,244,116,2,5.8,1.0,0.5,0.00,0.0,1.0,8.30,0.0,0.0,0.0


## Exploratory data analysis (EDA)

In [3]:
print("shape : ", df.shape)
print()
df.info(show_counts=True)

shape :  (3475226, 20)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3475226 entries, 0 to 3475225
Data columns (total 20 columns):
 #   Column                 Non-Null Count    Dtype         
---  ------                 --------------    -----         
 0   VendorID               3475226 non-null  int32         
 1   tpep_pickup_datetime   3475226 non-null  datetime64[us]
 2   tpep_dropoff_datetime  3475226 non-null  datetime64[us]
 3   passenger_count        2935077 non-null  float64       
 4   trip_distance          3475226 non-null  float64       
 5   RatecodeID             2935077 non-null  float64       
 6   store_and_fwd_flag     2935077 non-null  object        
 7   PULocationID           3475226 non-null  int32         
 8   DOLocationID           3475226 non-null  int32         
 9   payment_type           3475226 non-null  int64         
 10  fare_amount            3475226 non-null  float64       
 11  extra                  3475226 non-null  float64       
 12  mta_

In [4]:
print("\nMissing values : ")
df.isnull().sum()


Missing values : 


VendorID                      0
tpep_pickup_datetime          0
tpep_dropoff_datetime         0
passenger_count          540149
trip_distance                 0
RatecodeID               540149
store_and_fwd_flag       540149
PULocationID                  0
DOLocationID                  0
payment_type                  0
fare_amount                   0
extra                         0
mta_tax                       0
tip_amount                    0
tolls_amount                  0
improvement_surcharge         0
total_amount                  0
congestion_surcharge     540149
Airport_fee              540149
cbd_congestion_fee            0
dtype: int64

In [5]:
print("Percentage of missing values : ")
print(round((df.isnull().sum() / len(df)) * 100, 2))

Percentage of missing values : 
VendorID                  0.00
tpep_pickup_datetime      0.00
tpep_dropoff_datetime     0.00
passenger_count          15.54
trip_distance             0.00
RatecodeID               15.54
store_and_fwd_flag       15.54
PULocationID              0.00
DOLocationID              0.00
payment_type              0.00
fare_amount               0.00
extra                     0.00
mta_tax                   0.00
tip_amount                0.00
tolls_amount              0.00
improvement_surcharge     0.00
total_amount              0.00
congestion_surcharge     15.54
Airport_fee              15.54
cbd_congestion_fee        0.00
dtype: float64


In [6]:
print("Columns that have null values : ")
df.columns[df.isnull().any()].tolist()

Columns that have null values : 


['passenger_count',
 'RatecodeID',
 'store_and_fwd_flag',
 'congestion_surcharge',
 'Airport_fee']

In [7]:
df = df.drop(columns=["RatecodeID", "store_and_fwd_flag", "congestion_surcharge",
 "Airport_fee"])
print("Dropped Unnecessary columns")

Dropped Unnecessary columns


In [8]:
print("Needed columns :", df.columns)

Needed columns : Index(['VendorID', 'tpep_pickup_datetime', 'tpep_dropoff_datetime',
       'passenger_count', 'trip_distance', 'PULocationID', 'DOLocationID',
       'payment_type', 'fare_amount', 'extra', 'mta_tax', 'tip_amount',
       'tolls_amount', 'improvement_surcharge', 'total_amount',
       'cbd_congestion_fee'],
      dtype='object')


In [9]:
columns_to_keep = ['VendorID', 'tpep_pickup_datetime', 'tpep_dropoff_datetime',
       'passenger_count', 'trip_distance', 'PULocationID', 'DOLocationID',
       'payment_type', 'fare_amount', 'extra', 'mta_tax', 'tip_amount',
       'tolls_amount', 'improvement_surcharge', 'total_amount',
       'cbd_congestion_fee']

df_1 = pd.read_parquet(r'C:\prerit\Projects\Ride-Sharing-Demand-Intelligence\Raw\yellow_tripdata_2025-01.parquet', columns = columns_to_keep)
# df_1 = df_1.sample(1_500_000, random_state=42) # Dataset was very large, so for smooth execution i selected only 1.5 M records at a time
df_1.head()

,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,cbd_congestion_fee
0,1,2025-01-01 00:18:38,2025-01-01 00:26:59,1.0,1.60,229,237,1,10.0,3.5,0.5,3.00,0.0,1.0,18.00,0.0
1,1,2025-01-01 00:32:40,2025-01-01 00:35:13,1.0,0.50,236,237,1,5.1,3.5,0.5,2.02,0.0,1.0,12.12,0.0
2,1,2025-01-01 00:44:04,2025-01-01 00:46:01,1.0,0.60,141,141,1,5.1,3.5,0.5,2.00,0.0,1.0,12.10,0.0
3,2,2025-01-01 00:14:27,2025-01-01 00:20:01,3.0,0.52,244,244,2,7.2,1.0,0.5,0.00,0.0,1.0,9.70,0.0
4,2,2025-01-01 00:21:34,2025-01-01 00:25:06,3.0,0.66,244,116,2,5.8,1.0,0.5,0.00,0.0,1.0,8.30,0.0


In [10]:
df_1.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3475226 entries, 0 to 3475225
Data columns (total 16 columns):
 #   Column                 Dtype         
---  ------                 -----         
 0   VendorID               int32         
 1   tpep_pickup_datetime   datetime64[us]
 2   tpep_dropoff_datetime  datetime64[us]
 3   passenger_count        float64       
 4   trip_distance          float64       
 5   PULocationID           int32         
 6   DOLocationID           int32         
 7   payment_type           int64         
 8   fare_amount            float64       
 9   extra                  float64       
 10  mta_tax                float64       
 11  tip_amount             float64       
 12  tolls_amount           float64       
 13  improvement_surcharge  float64       
 14  total_amount           float64       
 15  cbd_congestion_fee     float64       
dtypes: datetime64[us](2), float64(10), int32(3), int64(1)
memory usage: 384.5 MB


In [11]:
df_1["passenger_count"] = df_1["passenger_count"].fillna(df_1["passenger_count"].median())

In [12]:
df_1.info(show_counts=True)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3475226 entries, 0 to 3475225
Data columns (total 16 columns):
 #   Column                 Non-Null Count    Dtype         
---  ------                 --------------    -----         
 0   VendorID               3475226 non-null  int32         
 1   tpep_pickup_datetime   3475226 non-null  datetime64[us]
 2   tpep_dropoff_datetime  3475226 non-null  datetime64[us]
 3   passenger_count        3475226 non-null  float64       
 4   trip_distance          3475226 non-null  float64       
 5   PULocationID           3475226 non-null  int32         
 6   DOLocationID           3475226 non-null  int32         
 7   payment_type           3475226 non-null  int64         
 8   fare_amount            3475226 non-null  float64       
 9   extra                  3475226 non-null  float64       
 10  mta_tax                3475226 non-null  float64       
 11  tip_amount             3475226 non-null  float64       
 12  tolls_amount           34752

### Checking for Invalid records

In [13]:
df_1 = df_1[df_1["trip_distance"] > 0]
df_1 = df_1[df_1["fare_amount"] > 0]
df_1 = df_1[df_1["total_amount"] > 0]
df_1 = df_1[df_1["passenger_count"] > 0]

In [14]:
print("Shape after removing invalid rows :", df_1.shape)

Shape after removing invalid rows : (3230197, 16)


### Creating new column "trip_duration_min"

In [15]:
df_1["trip_duration_min"] = round(((df_1["tpep_dropoff_datetime"] - df_1["tpep_pickup_datetime"]).dt.total_seconds() / 60), 2)

### removing invalid records(if any)

In [16]:
df_1 = df_1[df_1["trip_duration_min"] > 1]

## Feature extraction

In [17]:
df_1["pickup_hour"] = df_1["tpep_pickup_datetime"].dt.hour
df_1["pickup_day"] = df_1["tpep_pickup_datetime"].dt.day
df_1["pickup_weekday"] = df_1["tpep_pickup_datetime"].dt.weekday

Creating column for weekend

In [18]:
df_1["is_weekend"] = df_1["pickup_weekday"].isin([5,6]).astype(int)

### Removing Outliers

In [19]:
fare_low = df_1["fare_amount"].quantile(0.01)
fare_high = df_1["fare_amount"].quantile(0.99)
dist_low = df_1["trip_distance"].quantile(0.01)
dist_high = df_1["trip_distance"].quantile(0.99)

df_1 = df_1[(df_1["fare_amount"] >= fare_low) & (df_1["fare_amount"] <= fare_high)]
df_1 = df_1[(df_1["trip_distance"] >= dist_low) & (df_1["trip_distance"] <= dist_high)]

print("Shape after outlier removal:", df_1.shape)

Shape after outlier removal: (3132824, 21)


### Calculating Revenues 

In [20]:
df_1["revenue_per_mile"] = round(df_1["total_amount"] / df_1["trip_distance"], 2)
df_1["revenue_per_minute"] = round(df_1["total_amount"] / df_1["trip_duration_min"], 2)
df_1["tip_percentage"] = round(df_1["tip_amount"] / df_1["fare_amount"], 2)

## Sanity Check

In [21]:
df_1.describe()

,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,cbd_congestion_fee,trip_duration_min,pickup_hour,pickup_day,pickup_weekday,is_weekend,revenue_per_mile,revenue_per_minute,tip_percentage
count,3.132824e+06,3132824,3132824,3.132824e+06,3.132824e+06,3.132824e+06,3.132824e+06,3.132824e+06,3.132824e+06,3.132824e+06,3.132824e+06,3.132824e+06,3.132824e+06,3.132824e+06,3.132824e+06,3.132824e+06,3.132824e+06,3.132824e+06,3.132824e+06,3.132824e+06,3.132824e+06,3.132824e+06,3.132824e+06,3.132824e+06
mean,1.776666e+00,2025-01-17 10:38:11.581137,2025-01-17 10:52:56.394909,1.265234e+00,2.932790e+00,1.660278e+02,1.649150e+02,1.020892e+00,1.695316e+01,1.410361e+00,4.987893e-01,3.007151e+00,3.922020e-01,9.889106e-01,2.562967e+01,4.985604e-01,1.474690e+01,1.431889e+01,1.682585e+01,3.053180e+00,2.562748e-01,1.305746e+01,2.062322e+00,1.992453e-01
min,1.000000e+00,2024-12-31 20:47:55,2024-12-31 20:54:00,1.000000e+00,3.000000e-01,1.000000e+00,1.000000e+00,0.000000e+00,4.400000e+00,-3.250000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,5.800000e+00,0.000000e+00,1.020000e+00,0.000000e+00,1.000000e+00,0.000000e+00,0.000000e+00,3.500000e-01,0.000000e+00,0.000000e+00
25%,2.000000e+00,2025-01-10 07:45:50.750000,2025-01-10 08:01:23.750000,1.000000e+00,1.030000e+00,1.320000e+02,1.140000e+02,1.000000e+00,8.600000e+00,0.000000e+00,5.000000e-01,0.000000e+00,0.000000e+00,1.000000e+00,1.596000e+01,0.000000e+00,7.400000e+00,1.100000e+01,1.000000e+01,2.000000e+00,0.000000e+00,8.480000e+00,1.530000e+00,0.000000e+00
50%,2.000000e+00,2025-01-17 14:49:21.500000,2025-01-17 15:06:12,1.000000e+00,1.700000e+00,1.620000e+02,1.620000e+02,1.000000e+00,1.280000e+01,1.000000e+00,5.000000e-01,2.600000e+00,0.000000e+00,1.000000e+00,2.025000e+01,7.500000e-01,1.165000e+01,1.500000e+01,1.700000e+01,3.000000e+00,0.000000e+00,1.171000e+01,1.870000e+00,2.400000e-01
75%,2.000000e+00,2025-01-24 19:47:33,2025-01-24 20:01:53.250000,1.000000e+00,3.060000e+00,2.340000e+02,2.340000e+02,1.000000e+00,1.923000e+01,2.500000e+00,5.000000e-01,4.000000e+00,0.000000e+00,1.000000e+00,2.778000e+01,7.500000e-01,1.800000e+01,1.900000e+01,2.400000e+01,5.000000e+00,1.000000e+00,1.600000e+01,2.370000e+00,3.000000e-01
max,6.000000e+00,2025-02-01 00:00:44,2025-02-01 23:44:11,6.000000e+00,1.959000e+01,2.650000e+02,2.650000e+02,4.000000e+00,7.020000e+01,1.500000e+01,1.050000e+01,4.000000e+02,6.412000e+01,1.000000e+00,4.731500e+02,7.500000e-01,5.626320e+03,2.300000e+01,3.100000e+01,6.000000e+00,1.000000e+00,3.438000e+02,8.829000e+01,2.778000e+01
std,4.165803e-01,NaN,NaN,6.986982e-01,3.459508e+00,6.440358e+01,6.885167e+01,5.830943e-01,1.314462e+01,1.851535e+00,2.792880e-02,3.372858e+00,1.649264e+00,1.047157e-01,1.697912e+01,3.540591e-01,2.669672e+01,5.787116e+00,8.702759e+00,1.846713e+00,4.365754e-01,6.745883e+00,9.146716e-01,1.648196e-01


In [22]:
df_1.info()

<class 'pandas.core.frame.DataFrame'>
Index: 3132824 entries, 0 to 3475225
Data columns (total 24 columns):
 #   Column                 Dtype         
---  ------                 -----         
 0   VendorID               int32         
 1   tpep_pickup_datetime   datetime64[us]
 2   tpep_dropoff_datetime  datetime64[us]
 3   passenger_count        float64       
 4   trip_distance          float64       
 5   PULocationID           int32         
 6   DOLocationID           int32         
 7   payment_type           int64         
 8   fare_amount            float64       
 9   extra                  float64       
 10  mta_tax                float64       
 11  tip_amount             float64       
 12  tolls_amount           float64       
 13  improvement_surcharge  float64       
 14  total_amount           float64       
 15  cbd_congestion_fee     float64       
 16  trip_duration_min      float64       
 17  pickup_hour            int32         
 18  pickup_day             int3

## Saving Cleaned dataset

In [25]:
df_1.to_csv("Cleaned_yellow_tripdata_2025-01.csv", index=False)

In [ ]:
df_1.to_parquet("Cleaned_yellow_tripdata_2025-01.parquet", index=False)